# RAG Project – Recipe Question Answering System

## Introduction

This project implements a RAG system to answer questions about recipes.

The system retrieves relevant recipes from a dataset and generates answers using a language model.

Example queries:
- ingredients of a recipe
- how to cook a dish
- recipes containing a specific ingredient

## Data Description

- Dataset: recipe dataset from Kaggle  
- Each document contains:
  - recipe title  
  - ingredients list  
  - preparation steps  
- Data is structured to allow semantic retrieval and answer generation
- Link: [Kaggle Recipe Dataset](https://www.kaggle.com/datasets/wilmerarltstrmberg/recipe-dataset-over-2m)

## 1. Data Preparation

In [1]:
import pandas as pd
import ast

df = pd.read_csv("recipes_data.csv")

df = df[["title", "ingredients", "directions"]].dropna()
df = df.sample(5000, random_state=42).reset_index(drop=True)

def clean_list_text(x):
    if isinstance(x, str):
        try:
            value = ast.literal_eval(x)
            if isinstance(value, list):
                return "\n- " + "\n- ".join(str(v) for v in value)
        except:
            pass
    return str(x)

df["ingredients_clean"] = df["ingredients"].apply(clean_list_text)
df["directions_clean"] = df["directions"].apply(clean_list_text)

df["text"] = (
    "Recipe: " + df["title"].astype(str) +
    "\nIngredients:" + df["ingredients_clean"] +
    "\nSteps:" + df["directions_clean"]
)

print(df["text"].iloc[0])
print("Number of recipes:", len(df))

Recipe: Ice Cream Krispies
Ingredients:
- 12 cup butter
- 1 cup brown sugar
- 6 cups crisp rice cereal
- 1 cup coconut
- 12 cup chopped nuts
- 12 gallon vanilla ice cream, softened
- 16 ounces strawberries, cleaned, stemmed, & sliced
- 1 pint strawberry, cleaned and stemmed
- 13 cup white sugar
- 1 teaspoon vanilla
Steps:
- Prepare sauce: Cut strawberries in half.
- In a saucepan over medium high heat, combine strawberries, sugar and vanilla.
- Cook, stirring occasionally, until sauce thickens, about 5 minutes.
- Remove from heat.
- In a blender, puree sauce.
- Store sauce in refrigerator until ready to use.
- In large bowl, combine cereal, coconut and chopped nuts.
- Set aside.
- Combine butter and brown sugar in saucepan and warm until smooth.
- Pour over crispy rice mixture and stir.
- Place half of mixture on bottom of 9x13" pan and level.
- Smooth softened ice cream over top.
- Pour and level remaining mixture on top and freeze.
- When ready to serve, spoon strawberry sauce & stra

## 2. Embedding Generation


In [2]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_text(text):
    return embedding_model.encode(text, normalize_embeddings=True)

sample_embedding = embed_text(df["text"].iloc[0])

print("Embedding dimension:", len(sample_embedding))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


## 3. Vector Database



In [3]:
from chromadb import PersistentClient
from tqdm import tqdm

client = PersistentClient(path="./chroma_db")

try:
    client.delete_collection(name="recipes")
except:
    pass

collection = client.create_collection(
    name="recipes",
    metadata={"hnsw:space": "cosine"}
)

documents = df["text"].tolist()
metadatas = [{"title": df.loc[i, "title"]} for i in range(len(df))]

embeddings = []
ids = []

for i, doc in enumerate(tqdm(documents)):
    embeddings.append(embed_text(doc))
    ids.append(str(i))

collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=ids,
    metadatas=metadatas
)

print("Number of documents in DB:", collection.count())

100%|██████████| 5000/5000 [01:05<00:00, 75.78it/s] 

Number of documents in DB: 5000


## 4. Retrieval



In [4]:
def retrieve(query, n_results=3):
    query_embedding = embed_text(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    return results

query = "chicken with garlic"

results = retrieve(query, n_results=3)
context = "\n\n".join(results["documents"][0])
# Debug: show retrieved context
#print(context[:1200])

## 5. Answer Generation with LLM



In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cpu"

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_answer(question, context):

    PROMPT = f"""
Extract the recipe from the context below.

Do not explain.
Do not add anything.
Do not change the wording.

Return exactly:
Recipe:
Ingredients:
Steps:

<context>
{context}
</context>

Question: {question}
"""

    inputs = tokenizer(
        PROMPT,
        return_tensors="pt",
        truncation=True
    ).to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.pad_token_id
    )

    response = tokenizer.decode(output[0], skip_special_tokens=True)

    if "Recipe:" in response:
        response = response.split("Recipe:")[-1]
        response = "Recipe:" + response

    return response.strip()


q = "Give me a chicken recipe with garlic"

results = retrieve(q, n_results=1)
context = "\n\n".join(results["documents"][0])

answer = generate_answer(q, context)

print("QUESTION:", q)
print("\nANSWER:\n")
print(answer)

print("\nTITLES FOUND:")
for meta in results["metadatas"][0]:
    print("-", meta["title"])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: Give me a chicken recipe with garlic

ANSWER:

Recipe:
Ingredients:
- 1 small onion, diced
- 2 cups fresh mushrooms, sliced
- 1 garlic clove, minced
- 1/2 tsp. Dried thyme, divided
- 1 tablespoon butter or margarine
- 4 c. Cubed, leftover, cooked chicken
- 4 c. Leftover gravy
- 1 chicken bouillon cube, crushed
- Dash of pepper
- 3 cups Mashed Potatoes

Instructions:
1. In a large skillet over medium heat, sauté the onions, mushrooms, garlic, and 1/4 teaspoon thyme until softened (about 5 minutes).
2. Add the chicken, gravy, bouillon, and pepper to the skillet. Cook for about 8-10 minutes, stirring occasionally, until the chicken is no longer pink inside.
3. Pour the mixture into an oiled 3-quart casserole dish.
4. Bake at 375°F for 30-35 minutes, or until bubbling hot throughout.
5. Serve hot with your favorite side dishes. Enjoy!

TITLES FOUND:
- Chicken Feed


## 6. Evaluation



In [6]:
import time

def evaluate_answer(answer):
    score = 0
    
    if len(answer) > 100:
        score += 1
    
    if "Ingredients:" in answer and "-" in answer:
        score += 1
    
    if "Steps:" in answer and "-" in answer:
        score += 1
    
    return score


queries = [
    "chicken with garlic",
    "dessert",
    "pasta"
]

for q in queries:
    results = retrieve(q, n_results=1)
    context = "\n\n".join(results["documents"][0])

    start_time = time.time()
    answer = generate_answer(q, context)
    latency = time.time() - start_time

    score = evaluate_answer(answer)

    print("\nQUESTION:", q)
    print("SCORE:", score, "/3")
    print("LATENCY:", round(latency, 2), "seconds")

Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: chicken with garlic
SCORE: 3 /3
LATENCY: 4.77 seconds


Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: dessert
SCORE: 3 /3
LATENCY: 5.66 seconds

QUESTION: pasta
SCORE: 3 /3
LATENCY: 5.38 seconds


## 7. Experiments

In [7]:
queries_test = [
    "chicken with garlic",
    "quick pasta recipe"
]

for n in [1, 3]:
    print("\n====================")
    print(f"n_results = {n}")
    
    for q in queries_test:
        results = retrieve(q, n_results=n)
        context = "\n\n".join(results["documents"][0])
        answer = generate_answer(q, context)

        score = evaluate_answer(answer)

        print("\nQUESTION:", q)
        print("\nANSWER:\n", answer[:500])
        print("\nSCORE:", score, "/3")

Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



n_results = 1


Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: chicken with garlic

ANSWER:
 Recipe: Chicken Feed
Ingredients:
- 1 small onion, diced
- 2 c. fresh mushrooms, sliced
- 1 garlic clove, minced
- 1/2 tsp. dried thyme, divided
- 1 Tbsp. butter or margarine
- 4 c. cubed, leftover, cooked chicken
- 4 c. leftover gravy
- 1 chicken bouillon cube, crushed
- dash of pepper
- 3 c. mashed potatoes
Steps:
- In a skillet, saute the onion, mushrooms, garlic and 1/4 teaspoon thyme in butter. Stir in the chicken, gravy, bouillon and pepper. Spoon into a greased 3-quart casserole.
</contex

SCORE: 3 /3


Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: quick pasta recipe

ANSWER:
 Recipe: Three Cheese Pasta Bake
Ingredients:
- 12 cup unsalted butter, plus
- 2 tablespoons unsalted butter
- 13 cup all-purpose flour, plus
- 1 tablespoon all-purpose flour
- 5 cups milk
- 1 lb mild cheddar cheese, shredded (about 4 cups)
- 8 ounces Fontina cheese, rind removed, shredded
- 34 cup grated parmesan cheese
- 2 teaspoons Dijon mustard
- 2 teaspoons salt
- 12 teaspoon cayenne pepper
- 2 lbs penne rigate
- 1 cup fresh breadcrumb
- chopped flat leaf parsley, for garnish
Steps:
- Bring 

SCORE: 3 /3

n_results = 3


Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: chicken with garlic

ANSWER:
 Recipe: Penne With Lemon Chicken Strips
Ingredients:
- 8 ounces penne pasta
- 1 -3 tablespoon vegetable oil
- 1 small white onion, diced
- 1 lb chicken strips or 1 lb boneless chicken breast, diced
- 3/4 cup frozen corn
- 1/2 cup white wine
- 2 lemons, juice of
- 9 ounces spinach, roughly chopped
- 1/4 cup parmesan cheese, plus additional for serving
- 1/2 teaspoon fresh oregano, chopped
- salt and pepper, to taste
Steps:
- Cook pasta in lightly salted boiling water according to package directio

SCORE: 3 /3

QUESTION: quick pasta recipe

ANSWER:
 Recipe: Summer Pasta
Ingredients:
- 1 pt. cherry tomatoes, cut in half
- 3/4 c. olive oil (preferably extra virgin)
- 1/2 c. Parmesan cheese, freshly grated
- 10 oz. Mozzarella cheese, cubed
- 1/2 c. Romano cheese, freshly grated
- 2 to 3 cloves garlic, chopped
- handful of fresh basil, minced
- 1 lb. bow tie pasta or farfalle
- salt to taste
Steps:
- Saute garlic in olive oil over low heat, but don't 

## 8. Interface

In [8]:
import gradio as gr

def rag_system(question):
    results = retrieve(question, n_results=2)
    context = "\n\n".join(results["documents"][0])
    answer = generate_answer(question, context)
    return answer

interface = gr.Interface(
    fn=rag_system,
    inputs="text",
    outputs="text",
    title="Recipe RAG System",
    description="Ask for a recipe (e.g. chicken, pasta, dessert)"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Limitations and Conclusion

The system depends on the quality of retrieved documents, which directly impacts the generated answers.

Embedding quality also plays an important role in retrieval performance.

In some cases, answers may be incomplete due to context size limitations.

The system achieves good relevance as retrieved recipes match the user query.

Latency remains acceptable (around 5–6 seconds per query).

In this project, I built a complete RAG system that retrieves relevant recipes and generates structured answers.

Overall, combining retrieval and generation improves the relevance and reliability of responses compared to using a language model alone.